<a href="https://colab.research.google.com/github/it25101136/IT2011-Stress-Level-Prediction-/blob/Yehen/5_Marapana_KNN_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# BLOCK 1 - SHARED SETUP: Load the data and split it
# (Every group member's notebook starts with this same block)
# ============================================================

from google.colab import drive                          # lets Colab open files in your Google Drive
import pandas as pd                                     # pandas = tool for working with tables (like Excel in Python)
from sklearn.model_selection import train_test_split    # function that splits data into training / testing parts
import os                                               # basic file/folder tools (not really used below)

# Connect ("mount") Google Drive to Colab.
# Colab will show a pop-up asking you to allow access -> click Connect / Allow with your Google account.
drive.mount('/content/drive', force_remount=True)

# Read the cleaned dataset the group prepared earlier (already in the shared project folder).
# df_master = the whole table: 3000 students, one row per student.
df_master = pd.read_csv('/content/drive/MyDrive/IT2011_G12_AI_Project/results/outputs/FINAL_ML_READY_DATA.csv')

# FIX: Prevent Data Leakage
# Initially, the model hit 1.0 (100%) accuracy because it was "cheating".
# 'stress_score' is the direct numerical equivalent of the target 'stress_level'.
# Keeping it allowed the AI to read the answer key instead of learning patterns.
# By dropping it, we force the AI to predict stress using actual student habits
# (sleep, screen time, etc.). The accuracy drops to a realistic 0.69, proving
# the model is genuinely working and not suffering from data leakage.

# FIX: Drop BOTH the target category and the direct score to prevent data leakage
# X = the "clues" (input columns) the model is allowed to look at
# y = the "answer" we want the model to predict (stress_level: 0 / 1 / 2)
X = df_master.drop(columns=['stress_level', 'stress_score'])
y = df_master['stress_level']

# 80% Training, 20% Testing
# Training part = the model learns from these rows.
# Testing part  = hidden from the model, used at the end to check how well it learned.
# random_state=42 makes the split the SAME every time, so every member gets the same split (fair comparison).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Show the size of each part: (number of rows, number of columns)
# Expected: Training (2400, 7) and Testing (600, 7)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Mounted at /content/drive
Training data shape: (2400, 7)
Testing data shape: (600, 7)


In [3]:
# ============================================================
# BLOCK 2 - MEMBER 5 MODEL: KNN (K-Nearest Neighbours)
# Idea: to predict a student's stress level, find the K most similar
# students in the training data and take a "vote" of their stress levels.
# ============================================================

from sklearn.neighbors import KNeighborsClassifier         # the KNN model
from sklearn.model_selection import GridSearchCV           # tries many settings and picks the best one
from sklearn.metrics import classification_report, accuracy_score   # tools to measure how good the model is

print("Executing Member 5 Model: KNN")

# The settings ("hyperparameters") we want to try:
#   n_neighbors -> K = how many similar students to look at (3, 5, 7 or 9)
#   weights     -> 'uniform'  = every neighbour's vote counts the same
#                  'distance' = closer (more similar) neighbours count more
#   metric      -> how "distance / similarity" is measured
#                  'euclidean' = straight-line distance, 'manhattan' = grid/block distance
# 4 x 2 x 2 = 16 different combinations will be tested.
param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn = KNeighborsClassifier()   # create an empty KNN model

# GridSearchCV tests all 16 combinations.
# cv=5 -> for each combination, the training data is split into 5 pieces and tested 5 times (cross-validation),
#         so the choice is fair and not based on luck.
# scoring='accuracy' -> pick the combination with the highest % of correct predictions.
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)   # "fit" = train. Uses ONLY the training data.

best_model = grid_search.best_estimator_   # the KNN model with the best settings
y_pred = best_model.predict(X_test)        # predict stress level for the 600 test students it has never seen
accuracy = accuracy_score(y_test, y_pred)  # % of test predictions that were correct (e.g. 0.61 = 61%)

print(f"Optimum Parameters found: {grid_search.best_params_}")   # the winning settings
print(f"Validation Accuracy: {accuracy:.4f}")                    # accuracy to 4 decimal places

# Classification report = detailed score for each stress level (0, 1, 2):
#   precision -> when the model said "level X", how often it was right
#   recall    -> out of all real "level X" students, how many the model found
#   f1-score  -> a balance of precision and recall
#   support   -> how many test students actually had that level
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Save the final score into a small CSV file in the shared outputs folder.
# The group leader will combine everyone's Score_*.csv files to compare all models.
# This creates: results/outputs/Score_Marapana.csv  ->  columns: Model, Accuracy
pd.DataFrame({'Model': ['KNN'], 'Accuracy': [accuracy]}).to_csv(
    '/content/drive/MyDrive/IT2011_G12_AI_Project/results/outputs/Score_Marapana.csv', index=False)

print("Saved: results/outputs/Score_Marapana.csv")   # confirmation message

Executing Member 5 Model: KNN
Optimum Parameters found: {'metric': 'euclidean', 'n_neighbors': 7, 'weights': 'uniform'}
Validation Accuracy: 0.6100

Classification Report:
               precision    recall  f1-score   support

           0       0.46      0.21      0.29        62
           1       0.62      0.68      0.65       239
           2       0.62      0.64      0.63       299

    accuracy                           0.61       600
   macro avg       0.57      0.51      0.52       600
weighted avg       0.60      0.61      0.60       600

Saved: results/outputs/Score_Marapana.csv
